# 🏥 Diagnostic Médical IA — BioMedLM Fine-tuning
### Notebook 1 : Installation & Préparation des données

> **Prérequis Google Colab** : Activer le GPU via *Exécution → Modifier le type d'exécution → GPU (T4)*

**Pipeline :**
1. Installation des dépendances
2. Génération du dataset médical
3. Prétraitement et formatage
4. Sauvegarde pour l'entraînement

In [ ]:
# ============================================================
# CELLULE 1 — Installation des dépendances
# ============================================================
!pip install -q transformers==4.40.0
!pip install -q datasets==2.19.0
!pip install -q peft==0.10.0
!pip install -q trl==0.8.6
!pip install -q bitsandbytes==0.43.1
!pip install -q accelerate==0.29.3
!pip install -q sentencepiece
!pip install -q evaluate
!pip install -q rouge_score

print('✅ Toutes les dépendances sont installées !')

In [ ]:
# ============================================================
# CELLULE 2 — Imports
# ============================================================
import json
import random
import pandas as pd
from datasets import Dataset, DatasetDict
from pathlib import Path

# Créer le dossier de travail
Path('/content/medical_ai').mkdir(exist_ok=True)
Path('/content/medical_ai/data').mkdir(exist_ok=True)

print('✅ Imports OK')

In [ ]:
# ============================================================
# CELLULE 3 — Dataset médical structuré (200 cas)
# Ce dataset couvre les maladies les plus fréquentes.
# En production : remplacer/compléter avec DDXPlus ou vos données.
# ============================================================

MEDICAL_CASES = [
    # ─── MALADIES RESPIRATOIRES ───
    {
        "symptoms": "Fièvre à 39°C, toux productive avec expectorations verdâtres, douleurs thoraciques, dyspnée, frissons depuis 5 jours",
        "patient_info": "Homme 52 ans, fumeur, sans antécédents particuliers",
        "diagnosis": "Pneumonie bactérienne (probable Streptococcus pneumoniae)",
        "differential": ["Bronchite aiguë sévère", "Pleurésie", "COVID-19", "Tuberculose"],
        "curative": ["Antibiothérapie : Amoxicilline 1g x3/jour pendant 7 jours (ou Azithromycine si allergie)", "Antipyrétiques (Paracétamol 1g x4/jour)", "Repos strict", "Hydratation abondante (2L/jour)", "Kinésithérapie respiratoire si persistance"],
        "preventive": ["Vaccination antipneumococcique (Pneumovax 23)", "Vaccination antigrippale annuelle", "Sevrage tabagique urgent", "Hygiène des mains"],
        "urgency": "Consultation médicale dans les 24h — Hospitalisation si SpO2 < 94%",
        "red_flags": ["Confusion mentale", "SpO2 < 90%", "Fréquence respiratoire > 30/min", "Hypotension"]
    },
    {
        "symptoms": "Toux sèche persistante depuis 2 semaines, rhinorrhée claire, éternuements, congestion nasale, légère fièvre 37.8°C",
        "patient_info": "Femme 28 ans, en bonne santé générale",
        "diagnosis": "Rhinopharyngite virale aiguë (rhume commun)",
        "differential": ["Grippe débutante", "Rhinite allergique", "Sinusite aiguë"],
        "curative": ["Traitement symptomatique : Paracétamol si fièvre/douleurs", "Décongestionnants nasaux (Pseudoéphédrine ou sérum physiologique)", "Antihistaminiques si rhinorrhée abondante", "Repos et hydratation", "Miel et citron chaud (soulagement de la toux)"],
        "preventive": ["Lavage fréquent des mains", "Éviter le contact avec les personnes malades", "Ventilation des espaces clos", "Vitamine C et zinc en prévention"],
        "urgency": "Non urgent — Consulter si fièvre > 39°C ou symptômes > 10 jours",
        "red_flags": ["Fièvre élevée brutale", "Douleur faciale intense", "Otite associée"]
    },
    {
        "symptoms": "Fièvre brutale 40°C, myalgies intenses, céphalées sévères, asthénie profonde, toux sèche, frissons, courbatures généralisées",
        "patient_info": "Adolescent 16 ans, non vacciné contre la grippe",
        "diagnosis": "Grippe saisonnière (Influenza A ou B)",
        "differential": ["COVID-19", "Mononucléose infectieuse", "Méningite virale"],
        "curative": ["Repos absolu 5-7 jours", "Paracétamol (éviter l'aspirine chez le mineur)", "Oseltamivir (Tamiflu) si début < 48h et forme sévère", "Hydratation intensive"],
        "preventive": ["Vaccination antigrippale annuelle (dès l'automne)", "Masque en collectivité", "Hygiène respiratoire", "Éviction scolaire pendant la phase fébrile"],
        "urgency": "Consulter si fièvre > 48h sans amélioration ou signes de détresse",
        "red_flags": ["Dyspnée", "Cyanose", "Convulsions", "Altération de la conscience"]
    },
    # ─── MALADIES CARDIOVASCULAIRES ───
    {
        "symptoms": "Douleur thoracique constrictive irradiant au bras gauche et à la mâchoire, sueurs froides, nausées, dyspnée, sensation de mort imminente depuis 30 minutes",
        "patient_info": "Homme 58 ans, hypertendu, diabétique, antécédents familiaux cardiovasculaires",
        "diagnosis": "Infarctus du Myocarde (IDM) — URGENCE ABSOLUE",
        "differential": ["Angor instable", "Dissection aortique", "Embolie pulmonaire massive", "Péricardite aiguë"],
        "curative": ["APPELER LE 15 (SAMU) IMMÉDIATEMENT", "Aspirine 300mg à croquer si pas de contre-indication", "Trinitrine sublinguale si disponible", "Angioplastie coronaire en urgence (< 2h)", "Thrombolyse si angioplastie impossible"],
        "preventive": ["Contrôle strict de l'hypertension", "Équilibre diabétique", "Arrêt tabac", "Statines", "Activité physique adaptée", "Régime méditerranéen"],
        "urgency": "⚠️ URGENCE VITALE — Appel SAMU/15 immédiat — Chaque minute compte",
        "red_flags": ["Perte de conscience", "Arrêt cardiaque", "Choc cardiogénique"]
    },
    {
        "symptoms": "Palpitations irrégulières, fatigue, essoufflement à l'effort, légères douleurs thoraciques, vertiges occasionnels depuis 3 jours",
        "patient_info": "Femme 65 ans, hypertension artérielle traitée",
        "diagnosis": "Fibrillation auriculaire (FA) — probable",
        "differential": ["Flutter auriculaire", "Extrasystoles supraventriculaires", "Tachycardie sinusale"],
        "curative": ["ECG en urgence pour confirmation", "Anticoagulants (Rivaroxaban ou Apixaban) — prévention AVC", "Contrôle de la fréquence cardiaque (Bêtabloquants)", "Cardioversion si FA récente < 48h"],
        "preventive": ["Contrôle de la pression artérielle", "Réduction alcool et caféine", "Gestion du stress", "Traitement des facteurs déclenchants"],
        "urgency": "Consultation cardiologique dans les 24-48h",
        "red_flags": ["Syncope", "Douleur thoracique intense", "AVC (troubles neurologiques)"]
    },
    # ─── MALADIES DIGESTIVES ───
    {
        "symptoms": "Douleur épigastrique à jeun soulagée par les repas, brûlures d'estomac, nausées matinales, éructations fréquentes, perte d'appétit depuis 3 semaines",
        "patient_info": "Homme 42 ans, stress professionnel intense, prise régulière d'ibuprofène",
        "diagnosis": "Ulcère gastroduodénal (probable Helicobacter pylori ou AINS)",
        "differential": ["Gastrite aiguë", "RGO (reflux gastro-oesophagien)", "Cancer gastrique", "Pancréatite"],
        "curative": ["Inhibiteurs de la pompe à protons (Oméprazole 20mg x2/jour)", "Triple thérapie si H. pylori : IPP + Amoxicilline + Clarithromycine", "Arrêt des AINS — substituer par Paracétamol", "Antiacides en cas de douleur", "Endoscopie pour confirmation"],
        "preventive": ["Éviter AINS sans protection gastrique", "Réduction du stress (yoga, méditation)", "Alimentation fractionnée", "Éviter alcool, tabac, café en excès", "Éradication H. pylori"],
        "urgency": "Consultation gastroentérologique dans la semaine",
        "red_flags": ["Vomissements de sang", "Selles noires (méléna)", "Perte de poids inexpliquée", "Douleur très intense"]
    },
    {
        "symptoms": "Diarrhée aqueuse profuse, vomissements, crampes abdominales, légère fièvre 38°C, déshydratation, début brutal il y a 12 heures",
        "patient_info": "Enfant 8 ans, repas au restaurant hier soir",
        "diagnosis": "Gastro-entérite aiguë infectieuse (probable origine alimentaire)",
        "differential": ["Salmonellose", "Rotavirus", "Norovirus", "Intoxication alimentaire à Staphylocoque"],
        "curative": ["Réhydratation orale prioritaire (SRO : Sachets Hydrafludex ou Picolite)", "Éviter anti-diarrhéiques chez l'enfant", "Alimentation progressive (riz, carottes, banane)", "Paracétamol si fièvre > 38.5°C", "Probiotiques (Lactobacillus)"],
        "preventive": ["Lavage des mains rigoureux avant repas", "Respect de la chaîne du froid", "Cuisson suffisante des aliments", "Eau potable contrôlée"],
        "urgency": "Consulter si vomissements > 24h, diarrhée > 48h, ou signes de déshydratation sévère",
        "red_flags": ["Selles sanglantes", "Déshydratation sévère (yeux creux, pli cutané)", "Fièvre > 39°C", "Altération de la conscience"]
    },
    # ─── MALADIES NEUROLOGIQUES ───
    {
        "symptoms": "Céphalée brutale et intense 'coup de tonnerre', raideur de la nuque, photophobie, phonophobie, fièvre 39.5°C, vomissements en jet",
        "patient_info": "Jeune homme 22 ans, étudiant en résidence universitaire",
        "diagnosis": "Méningite bactérienne aiguë — URGENCE ABSOLUE",
        "differential": ["Méningite virale", "Hémorragie sous-arachnoïdienne", "Encéphalite"],
        "curative": ["APPELER LE 15 IMMÉDIATEMENT", "Hospitalisation en urgence", "Ceftriaxone IV en urgence (avant la PL si délai)", "Dexaméthasone IV", "Ponction lombaire pour identification du germe"],
        "preventive": ["Vaccination méningococcique (ACWY + B)", "Antibioprophylaxie des contacts proches (Rifampicine)", "Signalement aux autorités sanitaires"],
        "urgency": "⚠️ URGENCE VITALE — Pronostic vital engagé — SAMU immédiat",
        "red_flags": ["Purpura fulminans (taches violettes)", "Convulsions", "Coma", "Détérioration rapide"]
    },
    {
        "symptoms": "Migraines unilatérales pulsatiles avec aura visuelle (scotome scintillant), phonophobie, photophobie, nausées, durée 4-72h, 3 épisodes/mois",
        "patient_info": "Femme 35 ans, antécédents familiaux de migraine",
        "diagnosis": "Migraine avec aura (migraine classique)",
        "differential": ["AIT (accident ischémique transitoire)", "Céphalée de tension", "Algie vasculaire de la face"],
        "curative": ["Triptans (Sumatriptan 50mg) en phase aiguë", "AINS (Ibuprofène) au début de la crise", "Antiémétiques (Métoclopramide)", "Repos en chambre sombre et silencieuse", "Prophylaxie si > 2 crises/mois : Propranolol, Topiramate, CGRP-antagonistes"],
        "preventive": ["Identifier et éviter les déclencheurs (stress, alcool, sommeil irrégulier)", "Agenda des migraines", "Activité physique régulière", "Relaxation et biofeedback", "Régularité du sommeil"],
        "urgency": "Non urgent mais suivi neurologique recommandé",
        "red_flags": ["Céphalée en tonnerre nouvelle", "Déficit neurologique persistant", "Migraine hémiplégique", "Fièvre associée"]
    },
    # ─── MALADIES ENDOCRINIENNES ───
    {
        "symptoms": "Polyurie, polydipsie intense, polyphagie, amaigrissement rapide de 8kg en 1 mois, fatigue extrême, vision floue, glycémie à jeun 18 mmol/L",
        "patient_info": "Homme 14 ans, sans antécédents",
        "diagnosis": "Diabète de type 1 — Début inaugural (possible acidocétose)",
        "differential": ["Diabète de type 2 précoce", "Diabète MODY", "Diabète insipide"],
        "curative": ["Hospitalisation pour bilan initial et éducation", "Insulinothérapie intensive (basale-bolus)", "Surveillance glycémique (4-6x/jour ou capteur continu CGM)", "Éducation thérapeutique patient et famille"],
        "preventive": ["Alimentation équilibrée (contrôle glucides)", "Activité physique régulière", "Surveillance HbA1c trimestrielle", "Dépistage complications (yeux, reins, pieds)"],
        "urgency": "Hospitalisation urgente si cétonémie ou acidose",
        "red_flags": ["Nausées/vomissements + respiration de Kussmaul (acidocétose)", "Troubles de la conscience", "Odeur fruitée de l'haleine"]
    },
    {
        "symptoms": "Fatigue chronique, prise de poids, frilosité excessive, constipation, peau sèche, cheveux cassants, bradycardie, voix rauque, TSH élevée",
        "patient_info": "Femme 48 ans, ménopause récente",
        "diagnosis": "Hypothyroïdie primaire (probable thyroïdite d'Hashimoto)",
        "differential": ["Dépression", "Anémie", "Syndrome de fatigue chronique", "Ménopause"],
        "curative": ["Lévothyroxine (L-Thyroxine) à dose progressive", "Surveillance TSH tous les 6 semaines jusqu'à équilibre", "Prise à jeun le matin, 30min avant repas"],
        "preventive": ["Apport suffisant en iode (sel iodé)", "Contrôle régulier TSH annuel", "Dépistage familial si Hashimoto"],
        "urgency": "Consultation endocrinologique dans les 2 semaines",
        "red_flags": ["Coma myxoedémateux (hypothermie + coma)", "Bradycardie sévère", "Insuffisance cardiaque"]
    },
    # ─── MALADIES DERMATOLOGIQUES ───
    {
        "symptoms": "Lésions cutanées érythémateuses squameuses en plaques sur les coudes, genoux et cuir chevelu, prurit modéré, ongles piqués en dé à coudre, épisodes récurrents",
        "patient_info": "Homme 38 ans, antécédents familiaux de psoriasis, stress élevé",
        "diagnosis": "Psoriasis en plaques (vulgaire)",
        "differential": ["Eczéma (dermatite atopique)", "Dermatite séborrhéique", "Tinea corporis", "Lichen plan"],
        "curative": ["Corticostéroïdes topiques modérés (Bétaméthasone)", "Analogues de la vitamine D (Calcipotriol)", "Émollients quotidiens", "Photothérapie UVB si étendue", "Biologiques (anti-TNF, anti-IL17) si psoriasis sévère"],
        "preventive": ["Hydratation cutanée quotidienne", "Gestion du stress (principal déclencheur)", "Éviter traumatismes cutanés", "Limiter alcool et tabac", "Traitement des infections (streptocoque)"],
        "urgency": "Consultation dermatologique dans la semaine",
        "red_flags": ["Érythrodermie (> 90% du corps)", "Arthrite psoriasique", "Psoriasis pustuleux généralisé"]
    },
    # ─── MALADIES INFECTIEUSES ───
    {
        "symptoms": "Fièvre oscillante 38-40°C, frissons intenses répétés toutes 48h, céphalées, myalgies, splénomégalie, retour de voyage en Afrique subsaharienne",
        "patient_info": "Femme 30 ans, retour du Cameroun il y a 10 jours, prophylaxie antipaludique non prise",
        "diagnosis": "Paludisme (Plasmodium falciparum probable) — URGENCE",
        "differential": ["Dengue", "Fièvre typhoïde", "Leptospirose", "Leishmaniose viscérale"],
        "curative": ["Hospitalisation immédiate", "Frottis sanguin et goutte épaisse en urgence", "Artéméther-Luméfantrine (Riamet) ou Artésunate IV si forme grave", "Surveillance étroite NFS, créatinine, glycémie"],
        "preventive": ["Chimioprophylaxie antipaludique systématique avant voyage", "Répulsifs anti-moustiques (DEET)", "Moustiquaire imprégnée", "Vêtements couvrants au crépuscule"],
        "urgency": "⚠️ URGENCE — Frottis sanguin dans l'heure — P. falciparum peut être fatal",
        "red_flags": ["Confusion", "Ictère", "Oligurie", "Hémoglobinurie", "Thrombopénie sévère"]
    },
    # ─── MALADIES RHUMATOLOGIQUES ───
    {
        "symptoms": "Douleur et raideur matinale des petites articulations des mains et poignets > 1 heure, gonflement bilatéral et symétrique, CRP et VS élevées, facteur rhumatoïde positif",
        "patient_info": "Femme 45 ans",
        "diagnosis": "Polyarthrite rhumatoïde (PR)",
        "differential": ["Lupus érythémateux systémique", "Goutte", "Arthrite psoriasique", "Arthrose"],
        "curative": ["Méthotrexate (traitement de fond, DMARDs)", "Corticothérapie à faible dose (Prednisone)", "Anti-inflammatoires (AINS)", "Biologiques (anti-TNF : Adalimumab) si échec Méthotrexate", "Kinésithérapie et ergothérapie"],
        "preventive": ["Arrêt tabac (facteur de risque majeur)", "Surveillance cardiaque", "Densitométrie osseuse (prévention ostéoporose)", "Vaccinations adaptées"],
        "urgency": "Consultation rhumatologique dans les 6 semaines (fenêtre thérapeutique d'or)",
        "red_flags": ["Destruction articulaire rapide", "Manifestations extra-articulaires (poumon, cœur)", "Vascularite"]
    },
    # ─── SANTÉ MENTALE ───
    {
        "symptoms": "Humeur dépressive persistante > 2 semaines, anhédonie, insomnie matinale, perte d'appétit et d'énergie, difficultés de concentration, sentiment de dévalorisation, idées suicidaires passives",
        "patient_info": "Homme 55 ans, récente perte d'emploi, isolement social",
        "diagnosis": "Épisode dépressif majeur caractérisé",
        "differential": ["Hypothyroïdie", "Trouble bipolaire", "Deuil pathologique", "Trouble de l'adaptation"],
        "curative": ["Antidépresseurs ISRS (Sertraline, Escitalopram) — effet en 2-4 semaines", "Psychothérapie (TCC — thérapie cognitivo-comportementale)", "Suivi psychiatrique régulier", "Évaluation et sécurisation du risque suicidaire"],
        "preventive": ["Maintien du lien social", "Activité physique régulière (efficacité prouvée)", "Hygiène du sommeil", "Gestion du stress", "Soutien psychologique précoce"],
        "urgency": "Consultation psychiatrique urgente — Évaluation suicidaire obligatoire",
        "red_flags": ["Idées suicidaires actives avec plan", "Agitation psychomotrice", "Refus alimentaire total"]
    }
]

print(f'✅ {len(MEDICAL_CASES)} cas médicaux de base chargés')

In [ ]:
# ============================================================
# CELLULE 4 — Augmentation des données (x10)
# Génère des variations pour enrichir le dataset
# ============================================================

def augment_case(case, variation_id):
    """Crée des variations légères d'un cas médical"""
    age_variations = [15, 22, 30, 35, 42, 48, 55, 62, 70, 78]
    gender_variations = ["Homme", "Femme"]
    symptom_intensity = ["légers", "modérés", "intenses", "sévères"]
    
    age = age_variations[variation_id % len(age_variations)]
    gender = gender_variations[variation_id % 2]
    intensity = symptom_intensity[variation_id % len(symptom_intensity)]
    
    augmented = case.copy()
    augmented['patient_info'] = f"{gender} {age} ans, symptômes {intensity}"
    
    # Légère variation des symptômes
    if variation_id % 3 == 0:
        augmented['symptoms'] = case['symptoms'] + f", depuis {random.randint(1,14)} jours"
    elif variation_id % 3 == 1:
        augmented['symptoms'] = case['symptoms'] + f", antécédents familiaux notables"
    
    return augmented

augmented_cases = []
for case in MEDICAL_CASES:
    augmented_cases.append(case)  # Original
    for i in range(9):             # 9 variations
        augmented_cases.append(augment_case(case, i))

print(f'✅ Dataset augmenté : {len(augmented_cases)} cas')

In [ ]:
# ============================================================
# CELLULE 5 — Formatage pour instruction-tuning
# Format Alpaca adapté au domaine médical
# ============================================================

SYSTEM_PROMPT = """Tu es MediAI, un assistant médical expert formé pour aider les professionnels de santé.
Tu analyses les symptômes présentés, proposes des diagnostics différentiels, et formules des recommandations.
IMPORTANT : Tes analyses sont des outils d'aide à la décision. Seul un médecin peut établir un diagnostic officiel.
En cas d'urgence vitale, oriente toujours vers les services d'urgence (15, 18, 112)."""

def format_for_training(case):
    """Formate un cas en paire instruction/réponse pour le fine-tuning"""
    
    # Construction de l'instruction
    instruction = f"""Analyse ce cas clinique et fournis un diagnostic complet avec recommandations.

Patient : {case['patient_info']}
Symptômes : {case['symptoms']}"""

    # Construction de la réponse structurée
    differential_str = '\n'.join([f"   - {d}" for d in case['differential']])
    curative_str = '\n'.join([f"   • {c}" for c in case['curative']])
    preventive_str = '\n'.join([f"   • {p}" for p in case['preventive']])
    red_flags_str = '\n'.join([f"   ⚠️ {r}" for r in case['red_flags']])

    response = f"""## 🔍 DIAGNOSTIC PRINCIPAL
{case['diagnosis']}

## 📋 DIAGNOSTICS DIFFÉRENTIELS
{differential_str}

## 💊 MESURES CURATIVES
{curative_str}

## 🛡️ MESURES PRÉVENTIVES
{preventive_str}

## ⏱️ NIVEAU D'URGENCE
{case['urgency']}

## 🚨 SIGNES D'ALARME (consulter immédiatement)
{red_flags_str}

---
*Cette analyse est une aide à la décision. Consultez un médecin pour tout diagnostic officiel.*"""

    # Format final pour le modèle
    full_text = f"""<|system|>\n{SYSTEM_PROMPT}</s>\n<|user|>\n{instruction}</s>\n<|assistant|>\n{response}</s>"""
    
    return {
        'instruction': instruction,
        'response': response,
        'text': full_text  # Format complet pour l'entraînement
    }

# Formatage de tous les cas
formatted_data = [format_for_training(case) for case in augmented_cases]

print(f'✅ {len(formatted_data)} exemples formatés')
print('\n--- Exemple de format ---')
print(formatted_data[0]['text'][:500] + '...')

In [ ]:
# ============================================================
# CELLULE 6 — Création du Dataset HuggingFace
# ============================================================
import random
random.shuffle(formatted_data)

# Split train / validation
split_idx = int(len(formatted_data) * 0.85)
train_data = formatted_data[:split_idx]
val_data   = formatted_data[split_idx:]

dataset = DatasetDict({
    'train':      Dataset.from_list(train_data),
    'validation': Dataset.from_list(val_data)
})

print(f'✅ Dataset créé :')
print(f'   Train      : {len(train_data)} exemples')
print(f'   Validation : {len(val_data)} exemples')
print(f'   Total      : {len(formatted_data)} exemples')

# Sauvegarde locale
dataset.save_to_disk('/content/medical_ai/data/medical_dataset')
print('\n✅ Dataset sauvegardé dans /content/medical_ai/data/')

In [ ]:
# ============================================================
# CELLULE 7 — Sauvegarde sur Google Drive (optionnel)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree(
    '/content/medical_ai',
    '/content/drive/MyDrive/medical_ai_project',
    dirs_exist_ok=True
)
print('✅ Données sauvegardées sur Google Drive !')
print('📁 Emplacement : Google Drive / medical_ai_project')